# 내부 점검용 노트북
# 반출용 코드로 사용할 때는 df.head(), sample_records 제거 필요

In [1]:
# 노트북에서 src/utils/io.py를 불러올 수 있게 경로 잡기
# read_table_flexible() 바로 쓰기
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.io import read_table_flexible

In [2]:
# 환경 확인용
packages = ["pandas", "numpy", "geopandas", "shapely", "sklearn", "h3"]

for pkg in packages:
    try:
        module = __import__(pkg)
        print(f"{pkg}: OK / version={getattr(module, '__version__', 'unknown')}")
    except Exception as e:
        print(f"{pkg}: FAIL / {e}")

pandas: OK / version=2.3.3
numpy: OK / version=2.3.5
geopandas: FAIL / No module named 'geopandas'
shapely: FAIL / No module named 'shapely'
sklearn: FAIL / No module named 'sklearn'
h3: FAIL / No module named 'h3'


In [4]:
# 파일 읽기 테스트용
# 파일 하나 경로 넣고 실제로 읽히는지, 인코딩/구분자가 뭐로 잡히는지, 컬럼이 뭐가 있는지 확인
# B013 / B042 / B024 / B021 / B075 후보 중 하나씩 테스트
#test_path = Path("여기에_파일경로_입력.txt")  
test_path = Path(r"C:\Users\0215w\Downloads\B042 내국인(집계구) 성별연령대별.csv")

df, meta = read_table_flexible(test_path, nrows=1000)

print("읽기 성공")
print("encoding:", meta["encoding"])
print("sep:", repr(meta["sep"]))
print("shape:", df.shape)
print("columns:")
print(df.columns.tolist())

읽기 성공
encoding: cp949
sep: ','
shape: (500, 9)
columns:
['가맹점집계구코드(TOT_REG_CD)', '내국인업종코드(SB_UPJONG_CD)', '기준년월(TS_YM)', '일별(TS_YMD)', '개인법인구분(PSN_CPR)', '성별(SEX_CCD)', '연령대별(AGE_GB)', '카드이용금액계(AMT_CORR)', '카드이용건수(USECT_CORR)']


In [5]:
#데이터 앞부분 확인 - 반출 시에 해당 출력 셀 삭제할 것!!
df.head(3)

,가맹점집계구코드(TOT_REG_CD),내국인업종코드(SB_UPJONG_CD),기준년월(TS_YM),일별(TS_YMD),개인법인구분(PSN_CPR),성별(SEX_CCD),연령대별(AGE_GB),카드이용금액계(AMT_CORR),카드이용건수(USECT_CORR)
0,1122060020002,SB008,202103,20210319,개인,NaN,30대,41246.0,15.09
1,1118054020017,SB007,202003,20200331,개인,M,40대,40240.0,5.03
2,1102060020001,SB016,201807,20180711,법인,F,20대,11320518.0,5.03


In [6]:
# 컬럼/결측/자료형 빠르게 보기
print(df.dtypes)
print("\n결측치 개수 상위 20개")
print(df.isna().sum().sort_values(ascending=False).head(20))

가맹점집계구코드(TOT_REG_CD)       int64
내국인업종코드(SB_UPJONG_CD)     object
기준년월(TS_YM)                int64
일별(TS_YMD)                 int64
개인법인구분(PSN_CPR)           object
성별(SEX_CCD)               object
연령대별(AGE_GB)              object
카드이용금액계(AMT_CORR)        float64
카드이용건수(USECT_CORR)       float64
dtype: object

결측치 개수 상위 20개
연령대별(AGE_GB)             32
성별(SEX_CCD)              27
가맹점집계구코드(TOT_REG_CD)      0
내국인업종코드(SB_UPJONG_CD)     0
기준년월(TS_YM)               0
개인법인구분(PSN_CPR)           0
일별(TS_YMD)                0
카드이용금액계(AMT_CORR)         0
카드이용건수(USECT_CORR)        0
dtype: int64


---

In [7]:
# 좌표/공간 데이터 확인용
candidate_cols = [c for c in df.columns if any(k in c.upper() for k in ["X", "Y", "LON", "LAT", "COORD"])]
candidate_cols

['기준년월(TS_YM)', '일별(TS_YMD)', '성별(SEX_CCD)']

In [9]:
!pip install geopandas shapely

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 46.7 MB/s  0:00:00
   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   ----------------- ---------------------- 10.2/22.9 MB 50.0 MB/s eta 0:00:01
   ---------------------------------------  22.8/22.9 MB 54.7 MB/s eta 0:00:01
   ---------------------------------------- 22.9/22.9 MB 51.1 MB/s  0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 6.3/6.3 MB 53.7 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ---------- ----------------------------- 1/4 [pyproj]
   -------------------- ------------------- 2/4 [pyogrio]
   -------------------- ------------------- 2/4 [pyogrio]
   ------------------------------ --------- 3/4 [geopandas]
   ------------------------------ --------- 3/4 [geopandas]


In [ ]:
#geopandas import 및 geometry 생성 테스트
import geopandas as gpd
from shapely.geometry import Point

x_col = "X_COORD"   # 실제 컬럼명으로 수정
y_col = "Y_COORD"   # 실제 컬럼명으로 수정

sample_geo = df[[x_col, y_col]].dropna().head(100).copy()
sample_geo["geometry"] = sample_geo.apply(lambda r: Point(r[x_col], r[y_col]), axis=1)

gdf = gpd.GeoDataFrame(sample_geo, geometry="geometry", crs="EPSG:5179")  # 실제 좌표계에 맞게 수정
gdf.head()

# 좌표계(crs) 주의! 데이터 설명서 기반 : 
# B012 블록단위 지역경계도: EPSG:5181
# B024 블록단위 분기별 추정매출액의 블록 참조 shp: EPSG:5181
# B008/B010 SKT 유동인구 일부 좌표: EPSG:5179
# B009 KT 유동인구: EPSG:5186
# B013 버스정류장 좌표: EPSG:4326
# 파일 읽고 컬럼 보고, 좌표계도 같이 메모해야 함

In [ ]:
# 좌표계 변환 테스트
#나중에 H3 넣을 때 보통 WGS84 기준이 편해서 미리 변환해보는 셀
gdf_4326 = gdf.to_crs("EPSG:4326")
gdf_4326.head()

In [13]:
!pip install h3

   ---------------------------------------- 0.0/784.3 kB ? eta -:--:--
   ---------------------------------------- 784.3/784.3 kB 34.8 MB/s  0:00:00


In [14]:
# H3 테스트 셀 - h3 import가 되는 경우에만
import h3

In [ ]:
gdf_4326["lat"] = gdf_4326.geometry.y
gdf_4326["lng"] = gdf_4326.geometry.x

gdf_4326["h3_r8"] = gdf_4326.apply(
    lambda r: h3.latlng_to_cell(r["lat"], r["lng"], 8),
    axis=1
)

gdf_4326[["lat", "lng", "h3_r8"]].head()

---

In [15]:
# scripts/01_scan_dataset.py

from pathlib import Path
import json
import pandas as pd

from src.utils.io import read_table_flexible
from src.config import OUTPUT_DIR

def summarize_table(path: str):
    df, meta = read_table_flexible(path, nrows=1000)

    summary = {
        "file": str(path),
        "encoding": meta["encoding"],
        "sep": meta["sep"],
        "shape_preview": [int(df.shape[0]), int(df.shape[1])],
        "columns": df.columns.tolist(),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "null_counts_top20": df.isna().sum().sort_values(ascending=False).head(20).to_dict(),
        "sample_records": df.head(3).to_dict(orient="records"),
    }
    return summary

if __name__ == "__main__":
    import sys

    if len(sys.argv) < 2:
        raise SystemExit("사용법: python scripts/01_scan_dataset.py <파일경로>")

    out_dir = OUTPUT_DIR / "dataset_scans"
    out_dir.mkdir(parents=True, exist_ok=True)

    target = sys.argv[1]
    summary = summarize_table(target)

    out_path = out_dir / (Path(target).stem + "_summary.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"[완료] {out_path}")

RuntimeError: 파일을 읽지 못했습니다: --f=c:\Users\0215w\AppData\Roaming\jupyter\runtime\kernel-v355f588b63c74f4843235a7a08ab760e432e2187c.json
마지막 오류: [Errno 22] Invalid argument: '--f=c:\\Users\\0215w\\AppData\\Roaming\\jupyter\\runtime\\kernel-v355f588b63c74f4843235a7a08ab760e432e2187c.json'

In [16]:
# scripts/02_check_env.py

def try_import(name):
    try:
        module = __import__(name)
        return True, getattr(module, "__version__", "unknown")
    except Exception as e:
        return False, str(e)

if __name__ == "__main__":
    packages = ["pandas", "numpy", "geopandas", "shapely", "sklearn", "h3"]
    for pkg in packages:
        ok, info = try_import(pkg)
        print(f"{pkg}: {'OK' if ok else 'FAIL'} / {info}")

pandas: OK / 2.3.3
numpy: OK / 2.3.5
geopandas: OK / 1.1.3
shapely: OK / 2.1.2
sklearn: FAIL / No module named 'sklearn'
h3: OK / 4.4.2
